In [12]:
!pip install -q pymongo dnspython plotly pandas python-dotenv

import os
import json
from pymongo import MongoClient
from bson import json_util
from dotenv import load_dotenv

load_dotenv()

URI = os.getenv('MONGO_URI')
DB_NAME = os.getenv('DB_NAME', 'tracera')
COLLECTION_NAME = os.getenv('COLLECTION_NAME', 'raw_linkedin')

def connect_to_mongodb_and_get_data(uri, db_name, collection_name, query={}):
    if not uri:
        raise ValueError("Kredensial MONGO_URI tidak ditemukan! Pastikan file .env sudah ada.")
    try:
        client = MongoClient(uri)
        db = client[db_name]
        collection = db[collection_name]
        documents = list(collection.find(query))
        json_data = json.loads(json_util.dumps(documents))
        print(f"Berhasil terhubung! {len(json_data)} dokumen dimuat dari koleksi '{collection_name}'.")
        return json_data
    except Exception as e:
        print(f"Error koneksi: {e}")
        return []

raw_documents = connect_to_mongodb_and_get_data(URI, DB_NAME, COLLECTION_NAME)



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Berhasil terhubung! 397 dokumen dimuat dari koleksi 'raw_linkedin'.


In [13]:
import pandas as pd

# 1. Taksonomi Soft Skills Terkurasi (25 Kata Kunci Standar)
SOFT_SKILLS = {
    'communication', 'communications', 'public speaking', 'presentation skills', 'komunikasi',
    'teamwork', 'collaboration', 'interpersonal skills', 'kerjasama tim', 'active listening',
    'leadership', 'team leadership', 'management', 'strategic planning', 'kepemimpinan',
    'problem solving', 'analytical skills', 'critical thinking', 'decision making', 'creativity',
    'time management', 'adaptability', 'negotiation', 'english', 'teaching'
}

# 2. Parsing Dokumen Mentah & Pembersihan Data Anomali
def clean_doc(d):
    raw_list = d.get('raw_data', [{}])
    p = raw_list[0] if isinstance(raw_list, list) and len(raw_list) > 0 else {}

    # a. Validasi & ekstrak riwayat pengalaman kerja (rentang tahun 1980 - 2027)
    experiences = []
    years = []
    for e in p.get('experience', []):
        yr = e.get('startDate', {}).get('year') if isinstance(e.get('startDate'), dict) else None
        if yr and str(yr).isdigit():
            yr_int = int(yr)
            if 1980 <= yr_int <= 2027:
                years.append(yr_int)
                experiences.append({'year': yr_int, 'title': (e.get('position') or '').strip()})

    # b. Standarisasi nama industri (fallback ke 'Other' jika kosong/null)
    ind = p.get('industry')
    ind_name = ind.get('name') if isinstance(ind, dict) else (ind or 'Other')
    ind_clean = ind_name.strip() if isinstance(ind_name, str) and ind_name.strip() else 'Other'

    # c. Deduplikasi Keahlian per Profil (Case-insensitive & buang spasi liar)
    skill_map = {}
    all_raw_s = [s['name'] for s in p.get('skills', []) if isinstance(s, dict) and s.get('name')]
    all_raw_s += (p.get('topSkills') or [])
    for s in all_raw_s:
        if s and isinstance(s, str):
            s_clean = s.strip()
            if len(s_clean) > 1:
                key = s_clean.lower()
                if key not in skill_map:
                    skill_map[key] = s_clean
                elif s_clean[0].isupper() and not skill_map[key][0].isupper():
                    skill_map[key] = s_clean

    unique_skills = list(skill_map.values())

    # d. Bersihkan nama program studi (buang noise '-', '.', dan deduplikasi dalam 1 profil)
    progs = []
    for e in p.get('education', []):
        fos = e.get('fieldOfStudy')
        if fos and isinstance(fos, str):
            fos_clean = fos.strip()
            if len(fos_clean) > 2 and fos_clean not in ['-', '.', '...', 'null', 'None']:
                if fos_clean not in progs:
                    progs.append(fos_clean)

    return {
        'id': p.get('publicIdentifier'),
        'industry': ind_clean,
        'skills': unique_skills,
        'programs': progs,
        'experiences': experiences,
        'latest_work_year': max(years, default=None)
    }

# 3. Buat DataFrame Utama & Hapus Duplikasi Profil Alumni
raw_parsed = [clean_doc(d) for d in raw_documents]
df_raw = pd.DataFrame(raw_parsed)
df_profiles = df_raw.drop_duplicates(subset=['id']).dropna(subset=['id']).reset_index(drop=True)

# 4. Explode Skills ke Baris Individual & Tagging Soft Skills
df_skills = df_profiles.explode('skills').dropna(subset=['skills']).copy()
df_skills['skills_clean'] = df_skills['skills'].str.strip()
df_skills['is_soft'] = df_skills['skills_clean'].str.lower().isin(SOFT_SKILLS)

# 5. Ringkasan Hasil Preprocessing
n_soft = int(df_skills['is_soft'].sum())
n_hard = int((~df_skills['is_soft']).sum())

print("Selesai Preprocessing!")
print(f"- Data mentah masuk : {len(raw_documents)} dokumen")
print(f"- Profil bersih unik: {len(df_profiles)} alumni (profil scraping ganda dibersihkan)")
print(f"- Total baris skill : {len(df_skills)} baris")
print(f"  • Hard skills     : {n_hard} baris ({round(n_hard/len(df_skills)*100, 1)}%)")
print(f"  • Soft skills     : {n_soft} baris ({round(n_soft/len(df_skills)*100, 1)}%)")

Selesai Preprocessing!
- Data mentah masuk : 397 dokumen
- Profil bersih unik: 394 alumni (profil scraping ganda dibersihkan)
- Total baris skill : 6113 baris
  • Hard skills     : 5431 baris (88.8%)
  • Soft skills     : 682 baris (11.2%)


In [14]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

selected_industry = ""  # Contoh lain: "Mining", "Oil and Gas", atau "" (Semua)
top_n = 10

# Filter Data
df_s_filt = df_skills[df_skills['industry'].str.lower() == selected_industry.lower()] if selected_industry else df_skills

# Ambil Top N Hard & Soft Skills
top_hard = df_s_filt[~df_s_filt['is_soft']]['skills'].value_counts().head(top_n).sort_values(ascending=True)
top_soft = df_s_filt[df_s_filt['is_soft']]['skills'].value_counts().head(top_n).sort_values(ascending=True)

# Subplots 1 Baris, 2 Kolom
fig_a = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f"Top {top_n} Hard Skills", f"Top {top_n} Soft Skills"),
    shared_yaxes=False,
    horizontal_spacing=0.20
)

# Kiri: Hard Skills (Biru)
fig_a.add_trace(go.Bar(
    x=top_hard.values, y=top_hard.index,
    orientation='h', marker_color='#1d4ed8',
    name='Hard Skills', text=top_hard.values, textposition='auto'
), row=1, col=1)

# Kanan: Soft Skills (Hijau)
fig_a.add_trace(go.Bar(
    x=top_soft.values, y=top_soft.index,
    orientation='h', marker_color='#10b981',
    name='Soft Skills', text=top_soft.values, textposition='auto'
), row=1, col=2)

fig_a.update_layout(
    title_text=f"<b>a. Pemetaan Hard Skills & Soft Skills Alumni</b> — Industri: {selected_industry or 'Semua'}",
    height=420,
    showlegend=False,
    margin=dict(l=20, r=20, t=60, b=20)
)
fig_a.show()